# Data Quality — Companion Notebook

> Companion for `src/my_mlops_project/pipelines/data_quality/`.
> Shows the Great Expectations suite, the validation result, and the gate that
> halts the pipeline on a broken batch.

**Purpose:** validate the raw data against a quality contract and **fail loud**
if a batch breaks our assumptions (the first defence against silent model rot).

**Inputs:** `data/01_raw/...credit clients.xls`.
**Outputs:** `data/02_intermediate/validated_data.parquet`, `data/08_reporting/ge_report.json`.

## Table of Contents
1. [Setup](#1-setup)
2. [Load raw data](#2-load-raw-data)
3. [Explore: dictionary vs reality](#3-explore-dictionary-vs-reality)
4. [Run the expectation suite](#4-run-the-expectation-suite)
5. [The gate: halting a broken batch](#5-the-gate-halting-a-broken-batch)
6. [Notes for the report](#6-notes-for-the-report)

## 1. Setup

In [1]:
import sys
from pathlib import Path
import yaml
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
DATA = PROJECT_ROOT / "data"

params = yaml.safe_load(open(PROJECT_ROOT / "conf" / "base" / "parameters.yml"))["data_quality"]
params

{'expected_column_count': 25,
 'target_column': 'default payment next month',
 'ranges': {'AGE': {'min': 18, 'max': 100}, 'LIMIT_BAL': {'min': 0}},
 'value_sets': {'SEX': [1, 2],
  'EDUCATION': [0, 1, 2, 3, 4, 5, 6],
  'MARRIAGE': [0, 1, 2, 3],
  'default payment next month': [0, 1]},
 'not_null': ['ID', 'LIMIT_BAL', 'AGE', 'default payment next month']}

## 2. Load raw data

The original `.xls` has a 2-row header; `header=1` selects the real names (the raw file is never modified).

In [2]:
raw = pd.read_excel(
    DATA / "01_raw" / "Copy of default of credit card clients.xls",
    engine="xlrd", header=1,
)
print("shape:", raw.shape)
print((raw.head()).to_string())

shape: (30000, 25)
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default payment next month
0   1      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0                           1
1   2     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000                           1
2   3      90000    2          2         2   34      0      0      0      0      0      0      29239      14027      13559      14331      14948      15549      1518      1500      1000      1000      1000      5000                           0
3   4

## 3. Explore: dictionary vs reality

Before writing rules, look at what the data actually contains. The suite below
encodes valid ranges/sets; this is the evidence behind them — including the
**undocumented category codes** (`EDUCATION` 0/5/6, `MARRIAGE` 0) that the
quality step is designed to *detect*.

In [3]:
print("EDUCATION:", dict(raw["EDUCATION"].value_counts().sort_index()))
print("MARRIAGE :", dict(raw["MARRIAGE"].value_counts().sort_index()))
print("SEX      :", dict(raw["SEX"].value_counts().sort_index()))
print("AGE range:", raw["AGE"].min(), "-", raw["AGE"].max())

EDUCATION: {0: np.int64(14), 1: np.int64(10585), 2: np.int64(14030), 3: np.int64(4917), 4: np.int64(123), 5: np.int64(280), 6: np.int64(51)}
MARRIAGE : {0: np.int64(54), 1: np.int64(13659), 2: np.int64(15964), 3: np.int64(323)}
SEX      : {1: np.int64(11888), 2: np.int64(18112)}
AGE range: 21 - 79


## 4. Run the expectation suite

We call the production node so the notebook and pipeline cannot drift. It builds
the suite from `parameters.yml`, runs it, and returns a tidy report.

In [4]:
from my_mlops_project.pipelines.data_quality.nodes import validate_credit_data, gate_on_quality

validated, report = validate_credit_data(raw, params)
print("overall success:", report["success"], "| expectations:", report["n_expectations"], "| failed:", report["n_failed"])
print((pd.DataFrame(report["results"])).to_string())

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

overall success: True | expectations: 11 | failed: 0
                            expectation                      column  success
0    expect_table_column_count_to_equal                                 True
1    expect_column_values_to_be_between                         AGE     True
2   expect_column_values_to_not_be_null                         AGE     True
3    expect_column_values_to_be_between                   LIMIT_BAL     True
4   expect_column_values_to_not_be_null                   LIMIT_BAL     True
5     expect_column_values_to_be_in_set                         SEX     True
6     expect_column_values_to_be_in_set                   EDUCATION     True
7     expect_column_values_to_be_in_set                    MARRIAGE     True
8     expect_column_values_to_be_in_set  default payment next month     True
9   expect_column_values_to_not_be_null  default payment next month     True
10  expect_column_values_to_not_be_null                          ID     True


## 5. The gate: halting a broken batch

The whole point is to **stop** bad data. Here we deliberately corrupt a copy (an
impossible `AGE = -5`) and confirm the gate raises — this is what protects every
downstream pipeline.

In [5]:
broken = raw.copy()
broken.loc[0, "AGE"] = -5   # impossible age

_, broken_report = validate_credit_data(broken, params)
print("broken batch success:", broken_report["success"], "| failed:", broken_report["n_failed"])

try:
    gate_on_quality(broken_report)
except ValueError as e:
    print("\nGate correctly HALTED the pipeline:\n ", str(e)[:200])

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

broken batch success: False | failed: 1

Gate correctly HALTED the pipeline:
  Data quality gate FAILED — 1/11 expectations failed: ['expect_column_values_to_be_between(AGE)']. Pipeline halted before cleaning.


## 6. Notes for the report

> For [`../report/REPORT_OUTLINE.md`](../report/REPORT_OUTLINE.md) §3 (data quality).

- **Data unit tests (Great Expectations):** 11 expectations covering column
  count, numeric ranges (`AGE`, `LIMIT_BAL`), allowed category codes
  (`SEX`/`EDUCATION`/`MARRIAGE`/target), and non-null keys. All pass on the real
  data; the gate **raises on a broken batch** (demonstrated above).
- **It detected the undocumented category codes** (`EDUCATION` 0/5/6,
  `MARRIAGE` 0), which the `data_cleaning` step then consolidates — the two
  stages work together.
- **Production value:** validation runs as a Kedro node, so it can run on every
  incoming batch (`kedro run --pipeline=data_quality`) before training — a
  fail-fast gate against silent data corruption.